# Urban Mobility Analytics MVP
## Notebook 03: GTFS Processing & Public Transit Network Analysis

This notebook demonstrates how to process raw GTFS data to construct a network representation of the public transit system and analyze service supply patterns.

### Objectives:
1. **Load public transit schedules** (stops, routes, trips, edges) from processed Parquet files.
2. **Georeference stops** and map them onto city administrative boundaries (Comunas).
3. **Compute service frequency indicators** (trips per stop/commune).

---

### 1. Load Configurations & Data

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import yaml

with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Load stops, edges and comunas shapefile
stops_df = pd.read_parquet(os.path.join('..', config['gtfs']['stops_output']))
edges_df = pd.read_parquet(os.path.join('..', config['gtfs']['edges_output']))
comunas_gdf = gpd.read_file(os.path.join('..', config['paths']['comunas_shapefile']))

print(f"Stops: {stops_df.shape}")
print(f"Edges/Frequencies: {edges_df.shape}")

### 2. Spatial Mapping (Stops to Communes)

In [ ]:
# Convert stops to GeoDataFrame
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df['stop_lon'], stops_df['stop_lat']),
    crs='EPSG:4326'
)

# Ensure CRS match
if comunas_gdf.crs != 'EPSG:4326':
    comunas_gdf = comunas_gdf.to_crs('EPSG:4326')

# Spatial Join stops with communes
stops_with_communes = gpd.sjoin(stops_gdf, comunas_gdf, how='left', predicate='within')
print("Spatial join completed.")
stops_with_communes.head(3)

### 3. Calculating Commune-Level Supply KPIs

In [ ]:
# Find commune name column
commune_col = [c for c in comunas_gdf.columns if 'COMUNA' in c.upper()][0]

# 1. Count stops per commune
stops_count = stops_with_communes.groupby(commune_col).size().reset_index(name='stop_count')

# 2. Calculate average trip frequencies
# Join stops back with scheduled edges
edges_stops = edges_df.merge(stops_with_communes[[ 'stop_id', commune_col]], left_on='from_stop_id', right_on='stop_id')
frequency_sum = edges_stops.groupby(commune_col)['trip_frequency'].sum().reset_index(name='total_frequency')

# Combine metrics
commune_metrics = stops_count.merge(frequency_sum, on=commune_col)
commune_metrics

### 4. Supply Indicators Visualization

In [ ]:
# Merge metrics back to communes shapefile for plotting
plot_gdf = comunas_gdf.merge(commune_metrics, on=commune_col)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

plot_gdf.plot(column='stop_count', cmap='OrRd', legend=True, ax=axes[0], edgecolor='gray')
axes[0].set_title('Public Transit Stop Count per Commune')
axes[0].axis('off')

plot_gdf.plot(column='total_frequency', cmap='Purples', legend=True, ax=axes[1], edgecolor='gray')
axes[1].set_title('Scheduled Trips Frequency per Commune')
axes[1].axis('off')

plt.tight_layout()
plt.show()